# 第8回 ネットワークのしくみとインターネット ── 体験演習ノート

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

千葉大学 情報リテラシ（工学部1年）／オンデマンド回の **手を動かす演習** です。
スライドの内容を、自分の手でコードとして確かめます。**読むだけでなく、上から順にセルを実行**してください。

## このノートで体験すること
1. **IPアドレスと2進数** ── 10進⇔2進変換、`192.168.1.0/24` のネットワーク／ブロードキャスト／利用可能ホスト254台
2. **パリティ検査** ── 偶数パリティの付加と、1ビット誤りの検出
3. **シーザー暗号** ── 任意シフトでの暗号化・復号
4. **URL→表示の旅（録画デモ）** ── `dig` で名前解決、`socket` で名前→IP、`requests` でHTTP取得とヘッダ表示
5. **平文 vs HTTPS** ── なぜ公衆Wi-Fiで平文通信が危険か（ハガキと封筒のたとえ）
6. **データベースとSQL** ── `sqlite3`＋`pandas` で 選択・射影・結合
7. **データ分析** ── 代表値と外れ値、相関≠因果、ヒストグラム

## 使い方
- セルを選んで **Shift+Enter** で実行。上から順に。
- 🌐 **要ネット** と書いたセルは、インターネット接続が必要です（**Colab では最初からつながっています**）。手元のオフライン環境では動かない場合があります。
- 🎬 と書いた節は、**授業内で先生が実行して解説**します。皆さんは後から自分で再実行して追体験できます。

---

## 1. IPアドレスと2進数

IPv4アドレスは **32ビット**。8ビット（＝1バイト）ずつ4つに区切り、各かたまりを10進数 **0〜255** で表します（例：`192.168.1.10`）。
255 までしか使えないのは、8ビットで表せる数が \(2^8 = 256\) 個（=0〜255）だからです。

> たとえば `192` を2進数にすると `11000000`。各ビットには右から `1,2,4,8,16,32,64,128` の重みがあり、`128 + 64 = 192` です。

### 1-1. 10進 → 2進、2進 → 10進
まずは1つの数（0〜255）の変換を、桁の重みつきで見てみます。

In [ ]:
def to_bin8(n):
    """0〜255 の整数を 8ビットの2進文字列にする"""
    assert 0 <= n <= 255, "各オクテットは 0〜255"
    return format(n, "08b")

def show_weights(n):
    """桁の重み(128,64,...,1)と、その桁が1か0かを並べて表示する"""
    bits = to_bin8(n)
    weights = [128, 64, 32, 16, 8, 4, 2, 1]
    print(f"{n} を2進数にすると: {bits}")
    line_w = "  ".join(f"{w:>3}" for w in weights)
    line_b = "  ".join(f"{b:>3}" for b in bits)
    print("重み:", line_w)
    print("ビット:", line_b)
    used = [w for w, b in zip(weights, bits) if b == "1"]
    print("足し算:", " + ".join(map(str, used)) if used else "0", "=", n)

for v in [192, 168, 255, 1, 10]:
    show_weights(v)
    print("-" * 40)

# 2進 → 10進 も確認（int(文字列, 2)）
print("11000000 を10進に戻す:", int("11000000", 2))
print("11111111 を10進に戻す:", int("11111111", 2))

### 1-2. IPアドレス全体を2進数で見る
4つのオクテットをまとめて「ドット付き2進」で表示します。人間は10進、機械は2進で住所を扱っています。

In [ ]:
def ip_to_bin(ip):
    octets = [int(x) for x in ip.split(".")]
    assert len(octets) == 4 and all(0 <= o <= 255 for o in octets), "不正なIPv4"
    return ".".join(to_bin8(o) for o in octets)

for ip in ["192.168.1.10", "192.168.1.0", "192.168.1.255"]:
    print(f"{ip:>15}  ->  {ip_to_bin(ip)}")

### 1-3. `192.168.1.0/24` を手計算で分解する
`/24` は「左から **24ビットがネットワーク部**、残り **8ビットがホスト部**」という意味（サブネットマスク `255.255.255.0`）。

- **ネットワークアドレス**：ホスト部が全部0 → `192.168.1.0`
- **ブロードキャストアドレス**：ホスト部が全部1 → `192.168.1.255`
- **利用可能ホスト数**：\(2^8 - 2 = 254\) 台
  （ネットワークアドレスとブロードキャストアドレスの **2つは機器に割り当てられない** ので −2）

下のセルは、Python標準の `ipaddress` で答え合わせをします。

In [ ]:
import ipaddress

net = ipaddress.ip_network("192.168.1.0/24", strict=True)

print("対象ネットワーク     :", net)
print("サブネットマスク     :", net.netmask)
print("ネットワークアドレス :", net.network_address)
print("ブロードキャスト     :", net.broadcast_address)

# 利用可能ホスト（hosts() はネットワーク/ブロードキャストを除いた一覧）
hosts = list(net.hosts())
print("利用可能ホスト数     :", len(hosts), "台   ( = 2**8 - 2 =", 2**8 - 2, ")")
print("最初の割当可能IP     :", hosts[0])
print("最後の割当可能IP     :", hosts[-1])

# 手計算の式でも確認
host_bits = 32 - 24
print(f"\n手計算: ホスト部 = 32 - 24 = {host_bits} bit -> 2**{host_bits} - 2 = {2**host_bits - 2} 台")

## 2. パリティ検査（誤り検出）

通信の途中でノイズによりビットが反転することがあります。**パリティビット**は、データに1ビット足して
「1の個数を偶数に揃える（偶数パリティ）」ことで、**1ビットの反転を検出**するしくみです。
学生証や航空券の **チェックデジット** も同じ発想です。

- データ `1011001` の中の「1」の個数 = **4個（偶数）** → 偶数に保つためのパリティビット = **0**
- 送信ビット列 = `1011001` + `0` = `10110010`
- 受信側で「1の個数が奇数」なら、途中でビットが化けた＝**誤りを検出**

> ⚠ 注意：パリティは **1ビット誤りは検出できるが、2ビット同時に化けると見逃す**（偶数個の誤りに弱い）。位置の特定や訂正もできません。

In [ ]:
def parity_bit(data_bits, even=True):
    """data_bits: '1011001' のような文字列。偶数パリティのパリティビットを返す"""
    ones = data_bits.count("1")
    if even:
        return "0" if ones % 2 == 0 else "1"   # 1の個数を偶数に揃える
    else:
        return "1" if ones % 2 == 0 else "0"   # 奇数パリティ

def add_parity(data_bits, even=True):
    p = parity_bit(data_bits, even)
    return data_bits + p, p

data = "1011001"
sent, p = add_parity(data)
print(f"データ        : {data}（1の個数 = {data.count('1')} 個）")
print(f"パリティビット: {p}")
print(f"送信ビット列  : {sent}")

### 2-1. 受信側で誤りを検出する
受信したビット列の「1の個数」が偶数なら正常、奇数なら誤りと判定します。
わざと1ビットだけ反転させて、ちゃんと検出できるか試します。

In [ ]:
def check_even_parity(received):
    """受信ビット列(データ+パリティ)。1の個数が偶数なら True(正常)"""
    return received.count("1") % 2 == 0

print("送信ビット列          :", sent)
print("そのまま受信 -> 正常?  :", check_even_parity(sent), "（True=誤りなし）")

# 3ビット目を反転（1ビット誤り）させてみる
flipped = list(sent)
flipped[2] = "0" if flipped[2] == "1" else "1"
flipped = "".join(flipped)
print("\n1ビット化けた受信列    :", flipped)
print("検出結果 -> 正常?      :", check_even_parity(flipped), "（False=誤りを検出!）")

# 2ビット同時に反転（偶数個の誤り）は見逃すことも確認
flip2 = list(sent)
flip2[2] = "0" if flip2[2] == "1" else "1"
flip2[5] = "0" if flip2[5] == "1" else "1"
flip2 = "".join(flip2)
print("\n2ビット化けた受信列    :", flip2)
print("検出結果 -> 正常?      :", check_even_parity(flip2), "（True=見逃し… パリティの弱点）")

## 3. シーザー暗号（古典暗号・換字式）

アルファベットを一定数だけ「ずらす」換字式の暗号です（例：シフト3なら A→D, B→E …）。
鍵＝ずらす数。同じ数だけ逆にずらせば復号できます。

In [ ]:
def caesar(text, shift):
    """text を shift だけずらす（暗号化）。英大文字・小文字のみずらし、他はそのまま"""
    result = []
    for ch in text:
        if "A" <= ch <= "Z":
            result.append(chr((ord(ch) - ord("A") + shift) % 26 + ord("A")))
        elif "a" <= ch <= "z":
            result.append(chr((ord(ch) - ord("a") + shift) % 26 + ord("a")))
        else:
            result.append(ch)
    return "".join(result)

def caesar_decrypt(text, shift):
    return caesar(text, -shift)

plain = "Hello, Chiba University!"
key = 3
enc = caesar(plain, key)
dec = caesar_decrypt(enc, key)

print("平文      :", plain)
print(f"暗号文(+{key}):", enc)
print("復号      :", dec)
print("復号は平文と一致?:", dec == plain)

### 3-1. 鍵を知らないと？ ── 総当たり（ブルートフォース）
ずらし数は25通りしかないので、すべて試せば人間が読めてしまいます。
**古典暗号が現代では安全でない**理由です。

In [ ]:
secret = caesar("Meet at noon", 7)   # 鍵7で暗号化された文（鍵は秘密という設定）
print("暗号文:", secret, "\n")
print("--- 全シフトを試す（どれが意味の通る英文か探す）---")
for s in range(1, 26):
    print(f"shift={s:>2}: {caesar_decrypt(secret, s)}")

## 4. 🎬 録画デモ：URL → ページ表示までの「旅」

> 🎬 **ここは授業の中で先生が実行して解説する節です。** 皆さんは動画を見たあと、自分で同じセルを実行して追体験してください。
> 🌐 **要ネット**（Colab では最初からつながっています）。

ブラウザに `https://example.jp` と打ってからページが表示されるまでに、裏では次のことが起きています。

1. **名前解決（DNS）**：人間が覚える名前 `example.jp` を、機械の住所 **IPアドレス** に翻訳する
2. **接続**：そのIPアドレスのサーバへ TCP/IP でつなぐ
3. **取得（HTTP/HTTPS）**：「このページをください」と要求し、HTML が返ってくる

この旅を、Pythonとシェルコマンドで覗いてみます。

### 4-1. `dig` で名前解決を見る（DNS）
`dig` はDNSに問い合わせる定番ツール。`!` を付けるとColabのシェルでコマンドを実行できます。

- `+short` … 結果のIPアドレスだけを簡潔に表示
- `+trace` … **ルートDNS → TLD（.jp）→ 権威DNS** へと、問い合わせが委任されていく様子を1段ずつ表示

> 🎬 先生が `+trace` の出力を上から指でたどり、「ルートが .jp サーバを教え、.jp が example.jp の担当サーバを教える」という**委任のバケツリレー**を解説します。

In [ ]:
# 🌐 要ネット / 🎬 授業内デモ
# Colab に dig が無い場合に備えて入れておく（数秒）
!apt-get -qq install -y dnsutils > /dev/null 2>&1 || true

print("===== dig example.jp +short （IPアドレスだけ）=====")
!dig example.jp +short

In [ ]:
# 🌐 要ネット / 🎬 授業内デモ
print("===== dig +trace example.jp （ルート→TLD→権威DNS の委任を追う）=====")
!dig +trace example.jp

### 4-2. `socket` で「名前 → IP」をPythonでも解決する
シェルの `dig` と同じ名前解決を、Pythonの標準ライブラリ `socket` でも行えます。

In [ ]:
# 🌐 要ネット
import socket

for name in ["example.jp", "www.chiba-u.jp", "www.python.org"]:
    try:
        ip = socket.gethostbyname(name)
        print(f"{name:>16}  ->  {ip}")
    except OSError as e:
        print(f"{name:>16}  ->  解決できませんでした（{e}）")

### 4-3. `requests` でHTTPを取得し、ヘッダを表示する
最後に、そのサーバから実際にページ（HTML）を受け取ります。
**レスポンスヘッダ**には、サーバ名・コンテンツの種類・暗号化の有無などの「送り状」情報が入っています。

In [ ]:
# 🌐 要ネット / 🎬 授業内デモ
import requests

resp = requests.get("https://example.jp", timeout=10)

print("最終URL      :", resp.url, "（https に注目）")
print("ステータス   :", resp.status_code, resp.reason, "（200 OK = 成功）")
print("\n--- レスポンスヘッダ（サーバからの送り状）---")
for k, v in resp.headers.items():
    print(f"{k}: {v}")

print("\n--- 受け取ったHTMLの先頭500文字（これをブラウザが解釈して表示）---")
print(resp.text[:500])

> 🎬 先生はここで、`resp.text` のHTMLタグ（`<html>`, `<head>`, `<title>`, `<body>`）を指しながら
> 「人間が打った名前 → DNSでIPに翻訳 → サーバに接続 → HTMLが届く → ブラウザが見た目に変換」という
> **URL→表示の一本の旅**を振り返ります。

## 5. 平文 vs HTTPS ── なぜ公衆・野良Wi-Fiが危険か

カフェや空港の「無料Wi-Fi」では、**同じアクセスポイント**につながった他人が、流れる電波を拾える可能性があります。

| | たとえ | 中身 |
|---|---|---|
| **HTTP（平文）** | 🟥 **ハガキ** | 途中の配達人（同じWi-Fiの他人）が**中身を読める** |
| **HTTPS（暗号化）** | 🟩 **封筒** | 中身が**暗号化**され、途中で拾われても読めない |

URLの先頭が `https://`（鍵マーク🔒）なら、その通信は暗号化されています。
> 実際の盗聴（パケットキャプチャ）はこの演習では行いません。**概念**と、HTTPSの証明書情報を覗くデモで理解します。

参考：日本のフィッシング報告は2024年に過去最多 **171万件超**、その約75%が実在サービスの“なりすまし”。鍵マークとURL構造を読めることが最初の防具です。

### 5-1. HTTPS は本当に暗号化されているか ── 証明書を覗く
HTTPSサーバは「自分が本物である」ことを示す**デジタル証明書**を提示します。
これを発行・保証するのが**認証局（CA）**です。`ssl` でその中身を覗いてみます。

In [ ]:
# 🌐 要ネット
import ssl, socket

host = "example.jp"
ctx = ssl.create_default_context()
with ctx.wrap_socket(socket.socket(), server_hostname=host) as s:
    s.connect((host, 443))   # 443 は HTTPS のポート番号
    cert = s.getpeercert()

def name_str(seq):
    return ", ".join(f"{k}={v}" for t in seq for (k, v) in t)

print(f"接続先     : {host}:443 (HTTPS)")
print("証明書の所有者(subject):", name_str(cert.get("subject", [])))
print("発行者 CA  (issuer)  :", name_str(cert.get("issuer", [])))
print("有効期限   (notAfter):", cert.get("notAfter"))
print("\n=> この証明書があるので、ブラウザは『本物のサーバ』と確認し、通信を暗号化できる。")

### 5-2. http と https のヘッダを見比べる
同じ要求でも、`http://` と `https://` で何が変わるかをヘッダで確認します
（多くのサイトは http で来ても https に自動転送します）。

In [ ]:
# 🌐 要ネット
import requests

for url in ["http://example.jp", "https://example.jp"]:
    try:
        r = requests.get(url, timeout=10)
        # 実際にどのURLに落ち着いたか（http -> https へ転送されたか）
        print(f"要求: {url}")
        print(f"  最終URL : {r.url}")
        print(f"  暗号化  : {'あり (TLS)' if r.url.startswith('https') else 'なし (平文)'}")
        print(f"  Server  : {r.headers.get('Server', '(不明)')}")
        print("-" * 50)
    except Exception as e:
        print(f"{url}: 取得失敗 {e}")

## 6. データベースとSQL（選択・射影・結合）

リレーショナルデータベースは、データを**表（テーブル）**で管理します。
- 1行 = **レコード**、1列 = **フィールド（属性）**
- **SQL** で問い合わせる3つの基本操作：
  - **選択**：条件に合う **行** を取り出す（例：分類が「理工」の本だけ）
  - **射影**：必要な **列** だけ取り出す（例：書名と価格だけ）
  - **結合**：複数の表を、共通の列でつなぐ（例：書籍表 ＋ 著者表）

> ⚠ よくある間違い：「選択＝列を選ぶ」は誤り。**選択＝行、射影＝列**です。

ここでは標準ライブラリ `sqlite3`（小さなDB）と `pandas` を使います。

In [ ]:
import sqlite3
import pandas as pd

# メモリ上に小さなDBを作る
con = sqlite3.connect(":memory:")

# 著者テーブル
authors = pd.DataFrame({
    "著者ID": [1, 2, 3],
    "著者名": ["夏目漱石", "湯川秀樹", "牧野富太郎"],
})
# 書籍テーブル（著者IDで著者テーブルとつながる）
books = pd.DataFrame({
    "書籍ID": [101, 102, 103, 104],
    "書名":   ["こころ", "旅人", "植物知識", "三四郎"],
    "分類":   ["文学", "理工", "理工", "文学"],
    "価格":   [600, 880, 1200, 580],
    "著者ID": [1, 2, 3, 1],
})

authors.to_sql("著者", con, index=False, if_exists="replace")
books.to_sql("書籍", con, index=False, if_exists="replace")

print("=== 書籍テーブル ===")
print(books.to_string(index=False))
print("\n=== 著者テーブル ===")
print(authors.to_string(index=False))

### 6-1. 選択（行）・射影（列）
`WHERE` で行を絞り（選択）、`SELECT 列名` で列を選ぶ（射影）。

In [ ]:
# 選択: 分類が「理工」のレコード(行)だけ取り出す
print("【選択】分類 = 理工 の本")
print(pd.read_sql_query("SELECT * FROM 書籍 WHERE 分類 = '理工'", con).to_string(index=False))

# 射影: 書名と価格の列だけ取り出す
print("\n【射影】書名・価格 の列だけ")
print(pd.read_sql_query("SELECT 書名, 価格 FROM 書籍", con).to_string(index=False))

### 6-2. 結合（複数の表をつなぐ）
書籍表と著者表を `著者ID` でつないで、「どの本を誰が書いたか」を1つの表にします。

In [ ]:
query = """
SELECT 書籍.書名, 著者.著者名, 書籍.価格
FROM 書籍
JOIN 著者 ON 書籍.著者ID = 著者.著者ID
ORDER BY 書籍.価格 DESC
"""
print("【結合】書籍 × 著者")
print(pd.read_sql_query(query, con).to_string(index=False))

# 同じ結合を pandas の merge でも書ける（SQLとpandasの対応を見比べる）
print("\n【同じ結果を pandas の merge で】")
merged = books.merge(authors, on="著者ID")[["書名", "著者名", "価格"]]
print(merged.sort_values("価格", ascending=False).to_string(index=False))

## 7. データ分析 ── 代表値・外れ値・相関≠因果

ここからは `pandas` と `matplotlib` で、データの「読み方」を体験します。

### 代表値と外れ値
弁当の価格を、A町（5店）とB町（6店・1店だけ激安）で比べます。
- **平均値**は外れ値（極端な値）に**引っ張られやすい**
- **中央値（メジアン）**は外れ値の影響を**受けにくい**

In [ ]:
import pandas as pd
import numpy as np

A = pd.Series([260, 270, 280, 290, 300], name="A町")
B = pd.Series([100, 260, 270, 280, 280, 280], name="B町")  # 100円が外れ値

for s in [A, B]:
    print(f"{s.name}: {list(s)}")
    print(f"  平均値 = {s.mean():.1f} 円 /  中央値 = {s.median():.1f} 円 /  最頻値 = {s.mode().tolist()}")
    print()

print("=> B町は100円の店(外れ値)で平均が245円まで下がるが、中央値275円のほうが「真ん中の感覚」に近い。")

### 7-1. ヒストグラムで分布を見る
値のばらつきを階級ごとの本数（度数）で表したのがヒストグラムです。
> Colab上で図がそのまま表示されます。

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

# 日本語が文字化けする環境向けに、タイトル等は英語にしています
np.random.seed(0)
scores = np.random.normal(loc=65, scale=12, size=200).clip(0, 100)

plt.figure(figsize=(6, 4))
plt.hist(scores, bins=10, color="#3E78B2", edgecolor="white")
plt.axvline(scores.mean(), color="#A6192E", linestyle="--", label=f"mean = {scores.mean():.1f}")
plt.axvline(np.median(scores), color="#3C8A57", linestyle="-.", label=f"median = {np.median(scores):.1f}")
plt.title("Histogram of test scores (n=200)")
plt.xlabel("score")
plt.ylabel("count (frequency)")
plt.legend()
plt.tight_layout()
plt.show()

### 7-2. 相関 ≠ 因果
2つの量の関係の強さを示すのが**相関係数**（−1〜+1）。+1に近いほど強い正の相関です。
ただし **相関があっても因果があるとは限りません**。

有名な例：「アイスの売上が多い日ほど水難事故が多い」。
でも本当の原因は **気温（暑さ）** ──これを **交絡因子** と呼びます。アイスを禁止しても事故は減りません。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(1)
temp = np.linspace(15, 35, 40) + np.random.normal(0, 1.0, 40)   # 気温(交絡因子)
icecream = 5 * temp + np.random.normal(0, 8, 40)                 # アイス売上
accidents = 0.4 * temp + np.random.normal(0, 1.0, 40)            # 水難事故件数

df = pd.DataFrame({"temp": temp, "icecream": icecream, "accidents": accidents})

print("相関係数（−1〜+1）:")
print(f"  アイス売上 × 水難事故 : {df['icecream'].corr(df['accidents']):.3f}  （高い! でも因果ではない）")
print(f"  気温 × アイス売上     : {df['temp'].corr(df['icecream']):.3f}")
print(f"  気温 × 水難事故       : {df['temp'].corr(df['accidents']):.3f}")
print("\n=> 真の原因は『気温』。これが交絡因子。相関の裏に第3の要因がないか常に疑う。")

# 散布図 + 回帰直線
a, b = np.polyfit(df["icecream"], df["accidents"], 1)
plt.figure(figsize=(6, 4))
plt.scatter(df["icecream"], df["accidents"], color="#3E78B2", label="data")
xs = np.linspace(df["icecream"].min(), df["icecream"].max(), 100)
plt.plot(xs, a * xs + b, color="#A6192E", label=f"y = {a:.3f}x + {b:.2f}")
plt.title("Ice cream sales vs. water accidents (spurious correlation)")
plt.xlabel("ice cream sales")
plt.ylabel("water accidents")
plt.legend()
plt.tight_layout()
plt.show()

---
## 課題のヒント

ここまでの演習を踏まえ、Moodleの確認小テスト／レポートに取り組んでください。
手を動かして確かめた数字や図を、自分の言葉で説明できるようにするのが狙いです。

1. **サブネット**：`10.0.0.0/24` や `172.16.5.0/24` のネットワークアドレス・ブロードキャスト・利用可能ホスト数を、上の `ipaddress` のセルを書き換えて求めてみよう。`/24` を `/25` にすると何台になる？（ヒント：ホスト部が7ビットになる）
2. **パリティ**：データ `1110101` の偶数パリティビットは？ 自分で手計算してから `add_parity()` で答え合わせ。2ビット同時に化けたとき検出できないことも確かめよう。
3. **シーザー暗号**：友達と鍵（シフト数）を1つ決めて、短い英文を暗号化して交換し、復号してみよう。鍵を知らない場合は 5-1 の総当たりで読めてしまうことも体験。
4. **URL→表示の旅**：`socket.gethostbyname()` で、自分のよく使うサイトのIPを調べてみよう。`dig +trace` の出力で、どのサーバが委任を教えているか1段ずつ追ってみよう。（🌐要ネット）
5. **平文 vs HTTPS**：身近なサイトを `requests` で開き、`http://` が `https://` に転送されるか、最終URLで確認しよう。なぜ公衆Wi-Fiでログインする前に🔒を確認すべきか、ハガキ／封筒のたとえで説明してみよう。
6. **SQL**：書籍テーブルに自分の好きな本を1冊 `INSERT` で追加し、「価格が700円以上の本の書名と著者名」を結合＋選択＋射影で取り出すクエリを書いてみよう。
7. **データ分析**：身の回りのデータ（例：1週間の歩数や気温）を2つ用意し、相関係数を計算してみよう。相関があっても「本当に因果か？ 隠れた第3の要因はないか？」を必ず一言考えること。

> 困ったら：エラーメッセージをよく読む → 公式ドキュメントやAIに「このコードでこのエラーが出た、なぜ？」と**自分の理解を添えて**聞く → 友人・先生・院生LS・附属図書館も学びのモード。AIは便利な1つの道具であって、唯一の軸ではありません。